# Varying the Prompt: Effect on Accuracy

In [1]:
#%%
%load_ext autoreload
%autoreload 2

In [2]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import torch
import pandas as pd
import numpy as np
from functools import partial
from pathlib import Path
import logging
import random

from monoculture.analysis.setup import variations, RESULTS_ROOT_DIR, RESULTS_CSV_VARY_PROMPT, FIGURES_ROOT_DIR
from monoculture.analysis.utils import (
    key_to_model,
    get_size_and_it,
    load_model_outputs_same_prompt,
    get_metric,
    create_result_df,
    load_task_data,
    load_risk_scores,
    prettify_model_name,
    load_results_overview,
    load_data_if_needed, 
    add_evals_to_df,
    filter_results
)
from monoculture.analysis.metrics import (
    pairwise_agreement_at_random,
    matrix_pairwise_evals,
    pairwise_agreement_at_random,
    observed_agreement_wrapper,
    get_obs_agreement_counts,
)


from monoculture.analysis.plotting import (
    plot_agreement_lineplot,
    plot_recourse_lineplot,
    plot_recourse_lineplot_mean_stderr
)

from numpy.random import default_rng
rng = default_rng(seed=4304)

In [3]:
plot_config_file = "./results/.matplotlibrc"
plt.rcParams.update(mpl.rc_params_from_file(fname=plot_config_file))

FIGURES_PAPER_DIR = FIGURES_ROOT_DIR / "paper" / "same-prompt"
TASKS = ["ACSIncome"]
print("tasks", TASKS)
MODELS = [
    "Qwen--Qwen2.5-7B-Instruct",
    "Qwen--Qwen2.5-72B-Instruct",
    "meta-llama--Meta-Llama-3-8B-Instruct",
    "meta-llama--Meta-Llama-3.3-70B-Instruct",
]
variations

tasks ['ACSIncome']


{'feature_order': ['default', 'rand 1', 'rand 2', 'rand 3', ' reversed'],
 'format': ['bullet', 'text', 'comma'],
 'connector': ['is', '=', ':'],
 'granularity': ['original', 'low']}

## Load available models and variations

In [4]:
try:
    data_all
except NameError:
    print("'data_all' not yet defined")
    data_all = load_data_if_needed(data=None, tasks=TASKS)
else:
    print("Already defined, update if needed.")
    data_all = load_data_if_needed(data=data_all, tasks=TASKS)

'data_all' not yet defined
ACSIncome


## Load predictions

In [19]:
num_shots = 0
threshold_fitted = True

df = load_results_overview(
    num_shots=num_shots, threshold_fitted=threshold_fitted, same_prompt=False
)
df = df[df["task"].isin(TASKS)]
df = df[df["model"].isin(MODELS)]
df = add_evals_to_df(df)
df.shape

(383, 17)

In [20]:
try:
    assert set(TASKS).issubset(
        set(df["task"].unique())
    ), f"Results for all tasks to be analysed have to be available, available are: {list(df['task'].unique())} "
except AssertionError:
    TASKS = list(df["task"].unique())
    logging.warning(f"Reduced TASKS to available tasks: {TASKS}")

## Load Risk Scores, Predictions and Restrict to positive/negative instances

In [21]:
restrict_to_positive_label = True
restrict_to_negative_label = False
assert not (
    restrict_to_positive_label & restrict_to_negative_label
), "Choose one or none."


risk_scores = (
    df["predictions_path"]
    .apply(load_risk_scores)
    .apply(lambda x: x.squeeze())
    # .add_prefix("score_")
)
risk_scores.head()

if data_all:
    data = data_all.copy()
if any([restrict_to_positive_label, restrict_to_negative_label]):
    if not data_all:
        logging.warning("Load task data")
        data_all = load_task_data(TASKS, Path("./data"))
        data = data_all.copy()
    elif not all([t in data.keys()for t in TASKS]):
        for t in TASKS:
            if t not in data.keys():
                print(f'add {t} to data')
                data_t =load_task_data(t, Path("./data"))
                data_all.update(data_t)
                data = data_all.copy()

    for task in TASKS:
        df_task = df[df["task"] == task]
        y_true = data[task][1]

        if restrict_to_negative_label or restrict_to_positive_label:
            filter_idx = y_true[y_true == int(restrict_to_positive_label)].index
            risk_scores = risk_scores[filter_idx]
            data[task] = (data[task][0].loc[filter_idx], data[task][1].loc[filter_idx])
        print(task, risk_scores.shape)


df_with_riskscores = pd.concat(
    [df, risk_scores.add_prefix("score_")], axis=1
)
print(df_with_riskscores.shape)
display(df_with_riskscores.head())

df_with_predictions = df_with_riskscores.copy()
model_wise_thresholds = df_with_predictions["threshold"]
df_with_predictions.update(
    df_with_predictions.filter(like="score_").apply(
        lambda col: (col >= model_wise_thresholds).astype(int)
    )
)
df_with_predictions = df_with_predictions.rename(
    columns={
        col: col.replace("score_", "pred_", 1)
        for col in df.columns
        if col.startswith("score_")
    }
)

ACSIncome (383, 61233)
(383, 61250)


,task,model,is_inst,threshold_fitted,threshold,accuracy,balanced_accuracy,bench_hash,num_shots,prompt_format,...,score_1735597,score_1550578,score_2427891,score_2089348,score_608838,score_1995426,score_3187549,score_1588752,score_926337,score_688458
0,ACSIncome,Qwen--Qwen2.5-72B-Instruct,1,1,0.164835,0.761706,0.769047,175002779,0,text,...,0.014070,0.999739,0.997529,0.032967,0.999512,0.002188,0.999663,0.999568,0.999445,0.998831
1,ACSIncome,Qwen--Qwen2.5-72B-Instruct,1,1,0.148014,0.776888,0.772921,1344084919,0,bullet,...,0.007586,0.999770,0.946650,0.680272,0.999797,0.006691,0.999665,0.999796,0.999860,0.997210
2,ACSIncome,Qwen--Qwen2.5-72B-Instruct,1,1,0.003172,0.766957,0.779197,3156286684,0,text,...,0.001935,0.999703,0.106428,0.133005,0.999770,0.000804,0.995934,0.999512,0.999821,0.998678
3,ACSIncome,Qwen--Qwen2.5-72B-Instruct,1,1,0.003183,0.772761,0.776203,2985942201,0,bullet,...,0.001936,0.999569,0.067419,0.377005,0.999841,0.000430,0.992414,0.999447,0.999877,0.999089
4,ACSIncome,Qwen--Qwen2.5-72B-Instruct,1,1,0.007574,0.779946,0.777026,1338756957,0,bullet,...,0.001922,0.999619,0.222584,0.438320,0.999860,0.000553,0.995919,0.999704,0.999891,0.998967


In [22]:
for model in df_with_predictions['model'].unique():
    print(model,'\t', df_with_predictions[df_with_predictions['model'] == model].shape[0])

Qwen--Qwen2.5-72B-Instruct 	 77
meta-llama--Meta-Llama-3-8B-Instruct 	 105
Qwen--Qwen2.5-7B-Instruct 	 90
meta-llama--Meta-Llama-3.3-70B-Instruct 	 111


In [23]:
models_to_plot = MODELS
models_to_plot = sorted(models_to_plot, key=get_size_and_it)

## Agreement between models

In [24]:
df_sampe_prompt = load_results_overview(
    num_shots=num_shots, threshold_fitted=threshold_fitted, same_prompt=True
)
df_sampe_prompt = df_sampe_prompt[df_sampe_prompt["task"].isin(TASKS)]
df_sampe_prompt.shape

(50, 15)

In [25]:
predictions_same_prompt = load_model_outputs_same_prompt(
    df_sampe_prompt, tasks=TASKS, return_risk_scores=False
)

df_sampe_prompt = add_evals_to_df(df_sampe_prompt)

ACSIncome


In [26]:
predictions_same_prompt, data = filter_results(
    predictions=predictions_same_prompt.copy(),
    df=df,
    data={t: val for t, val in data_all.items() if t in TASKS},
    tasks=TASKS,
    restrict_to_positive_label=restrict_to_positive_label,
    restrict_to_negative_label=restrict_to_negative_label,
    acc='accuracy'
)

Using accuracy for comparison.
ACSIncome (61233, 4)
ACSIncome (61233, 4)


compute agreement matrix for same prompt

In [27]:
models_sorted = sorted(predictions_same_prompt[TASKS[0]].columns, key=get_size_and_it)
M_same_prompt = len(models_sorted)
# agreements without diagonal
obs_agreement_matrix_same_prompt = matrix_pairwise_evals(
    models_sorted,
    partial(observed_agreement_wrapper, predictions=predictions_same_prompt[TASKS[0]]),
).fill_diagonal_(torch.nan)

observed_same_prompt = obs_agreement_matrix_same_prompt[
    ~obs_agreement_matrix_same_prompt.isnan()
]
M_pairs_same_prompt = len(observed_same_prompt)
recourse_fraction_models_same_prompt = get_obs_agreement_counts(
    predictions_same_prompt[TASKS[0]],
    restrict_only_pos_instances=True,
    true_labels=data[TASKS[0]][1],
)
recourse_fraction_models_same_prompt = sorted(
    recourse_fraction_models_same_prompt / predictions_same_prompt[TASKS[0]].shape[1]
)

## Agreement and Recourse

In [ ]:
default_width = plt.rcParams['figure.figsize'][0]*0.9
# Set custom height
custom_height = 2.3 #plt.rcParams['figure.figsize'][1]

fig, axs = plt.subplots(
    2,
    len(models_to_plot),
    figsize=(default_width, custom_height),
    constrained_layout=True,
    gridspec_kw={'height_ratios': [1,2]},
    sharey="row",
)
legend = True

recourse_observed = {}
recourse_at_random = {}
for m, model in enumerate(models_to_plot[:1]):
    print(model)

    predictions_model_variations = (
            df_with_predictions[df_with_predictions["model"] == model]
            .filter(like="score_")
            .transpose()
        )
    predictions_model_variations.index = predictions_model_variations.index.str.replace(
            r"^score_", "", regex=True
        ).astype(int)

    for i in range(2):
        ax = axs[i, m]
        if i == 0: # plot agreement
            # for reference plot agreement same prompt, varying models
            fraction_model_pairs = np.arange(0, M_pairs_same_prompt - 1 + 0.01, 1.0) / (
                M_pairs_same_prompt - 1
            )
            ax.plot(
                fraction_model_pairs,
                sorted(observed_same_prompt),
                label="vary model",
                color="gray",
                zorder=-2
            )

            # compute agreement for 10 equally sized variation sets
            variation_indices = predictions_model_variations.columns
            num_samples = 20
            sampled_variation_indices = [
                random.sample(list(variation_indices), M_same_prompt)
                for _ in range(num_samples)
            ]
            agreements_obs = []
            agreements_rand = []
            for sample in sampled_variation_indices:
                obs_agreement_matrix = matrix_pairwise_evals(
                    sample,
                    partial(
                        observed_agreement_wrapper, predictions=predictions_model_variations
                    ),
                ).fill_diagonal_(torch.nan)
                exp_agreement_matrix = matrix_pairwise_evals(
                    sample,
                    fun=lambda m1, m2: pairwise_agreement_at_random(
                        df[df["model"] == model].loc[m1]["accuracy"],
                        df[df["model"] == model].loc[m2]["accuracy"],
                    ),
                ).fill_diagonal_(torch.nan)
                agreements_obs.append(
                    torch.Tensor(sorted(obs_agreement_matrix[~obs_agreement_matrix.isnan()]))
                )
                agreements_rand.append(
                    torch.Tensor(sorted(exp_agreement_matrix[~exp_agreement_matrix.isnan()]))
                )

            agreements_obs = torch.stack(agreements_obs)
            agreements_rand = torch.stack(agreements_rand)
            stderr = agreements_obs.std(dim=0) / np.sqrt(num_samples)
            ax = plot_agreement_lineplot(
                ax,
                agreements_obs.mean(dim=0),
                agreements_rand.mean(dim=0),
                title=f"{prettify_model_name(key_to_model(model))}",
                # ylabel="agreement",
                # xlabel="fraction of variation pairs",
            )
            ax.fill_between(
                fraction_model_pairs,
                agreements_obs.mean(dim=0) - stderr,
                agreements_obs.mean(dim=0) + stderr,
                color="C0",
                alpha=0.7,
            )
        else: # plot recourse
            tp_rates = [
                [
                    1.0 - df[df["model"] == model].loc[var_idx]["fnr"]
                    for var_idx in sample
                ]
                for sample in sampled_variation_indices
            ]

            ax, obs, at_rand = plot_recourse_lineplot_mean_stderr(
                ax,
                [predictions_model_variations[sample] for sample in sampled_variation_indices],
                baseline_rates=tp_rates,
                y_true=data[task][1],
                restrict_only_pos_instances=True,
                restrict_only_neg_instances=False,
                count_accepted=True,
                at_least=False,
                plot_pdf=True,
                show_monoc=True,
                xlabel="",
                ylabel="",
                num_ticks = 3,
            )
            recourse_observed[model] = obs
            recourse_at_random[model] = at_rand

axs[0, 0].set_ylabel("agreement")
axs[0, 0].set_xlabel("fraction of variation pairs")

axs[1, 0].set_ylabel("fraction of\n variations accepting")
axs[1, 0].set_xlabel("fraction of positive instances")


if legend: 
    # legend
    desired_order = ['observed', 'random error', 'random prediction', 'monoculture']
    handles, labels = axs[0,0].get_legend_handles_labels()
    unique = {}
    for h, l in zip(handles, labels):
        if l not in unique:
            unique[l] = h
    ordered_labels = [label for label in desired_order if label in unique]
    ordered_handles = [unique[label] for label in ordered_labels]
    # positioning
    # get the bottom-most y-position among all axes
    lowest_ax_y = min([ax.get_position().y0 for ax in axs.flat])

    fig.legend(
        ordered_handles,
        ordered_labels,
        loc="upper left",
        # ncol=len(labels),
        bbox_to_anchor=(1.0, 1.0),
        # loc="lower center",
        # ncol=len(labels),
        # bbox_to_anchor=(0.5, lowest_ax_y-0.22),
        frameon=False,  # show box
        framealpha=1.0,  # opacity of box
        edgecolor="black",  # border color
        fancybox=True,
    )  # rounded corners)

file_name = "".join(
    [
        f"agreement-recourse-all-variations-with-same-prompt-agreement-{num_shots}-shot",
        "_tresh_fitted" if threshold_fitted == 1 else "",
        "_pos_instances" if restrict_to_positive_label else "",
        "_neg_instances" if restrict_to_negative_label else "",
    ]
)
# for ending in [".png", ".pdf"]:
    # plt.savefig(FIGURES_PAPER_DIR / (file_name + ending))
print(file_name)

plt.show()

In [161]:
print(f"{'model':15}&\tno recourse & limited recourse (majority rejects) & full recourse\\\\")
for model in recourse_observed.keys():
    print(f"{prettify_model_name(key_to_model(model)):15} &\t{recourse_observed[model][recourse_observed[model][:,1] == 0.0][-1][0]:.4f}&\t{recourse_observed[model][recourse_observed[model][:,1] <= 0.5][-1][0]:.4f} &\t{1. - recourse_observed[model][recourse_observed[model][:,1] == 1.0][0][0]:.4f}\\\\")

model          &	no recourse & limited recourse (majority rejects) & full recourse\\
Qwen 2.5 7B (it) &	0.1132&	0.2463 &	0.6078\\
Llama 3 8B (it) &	0.1079&	0.2329 &	0.5900\\
Llama 3.3 70B (it) &	0.0834&	0.1776 &	0.6738\\
Qwen 2.5 72B (it) &	0.0838&	0.1954 &	0.6350\\
